In [1]:
# ══════════════════════════════════════════════════════════════════
# NOTEBOOK 1 (REBUILT): DATA PREPARATION — PROPHET MODEL
# Approach: Compare current values vs fixed 5Y baseline average
#
# Baseline: Jan 2021 → Dec 2025 (60 months)
# Current:  Jan 2026 → Jun 2026 (actual data)
# Forecast: Jul 2026 → Sep 2026 (Prophet will produce)
#
# For each indicator:
#   1. Load raw source data
#   2. Compute fixed 5Y baseline average
#   3. Compute deviation or ratio vs baseline
#   4. This deviation/ratio becomes the Prophet input
#
# Indicators:
#   STU:    deviation = current - baseline_avg
#   BDI:    ratio     = current / baseline_avg
#   Demand: ratio     = current / baseline_avg
#   PPI:    ratio     = current / baseline_avg
#   KSA:    deviation = current - baseline_avg (%)
#   Policy: rule-based — no baseline needed
#
# Output tables:
#   srm.prophet_ts_stu
#   srm.prophet_ts_demand
#   srm.prophet_ts_ppi
#   srm.prophet_ts_ksa
#   srm.prophet_ts_policy
#   srm.prophet_ts_bdi
#   srm.prophet_ts_all
#   srm.prophet_baseline_summary  ← new: baseline stats
# ══════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

COMMODITIES   = ["Wheat","Corn","Rice","Soybean", "Barley"]
BASELINE_START = pd.Timestamp("2021-01-01")
BASELINE_END   = pd.Timestamp("2025-12-01")
SCORING_END    = pd.Timestamp("2026-07-01")

STRING_COLS = [
    "ksa_top1_country","ksa_top1_country_lag1",
    "ksa_top3_countries","top_buyer_country",
    "individually_tracked_countries"
]

def save_to_lakehouse(df_pandas, table_name, schema="srm"):
    full_name = f"{schema}.{table_name}"
    spark.createDataFrame(df_pandas) \
         .write.mode("overwrite") \
         .option("overwriteSchema","true") \
         .format("delta") \
         .saveAsTable(full_name)
    count = spark.table(full_name).count()
    print(f"✓ {full_name}: {count} rows saved")

def load_table(table_name, drop_strings=False):
    df_spark = spark.table(f"srm.{table_name}")
    if drop_strings:
        drop_cols = [c for c in STRING_COLS if c in df_spark.columns]
        if drop_cols:
            df_spark = df_spark.drop(*drop_cols)
    df = df_spark.toPandas()
    if "year_month" in df.columns:
        df["year_month"] = pd.to_datetime(df["year_month"])
    return df

print("=== Notebook 1 (Rebuilt): Data Preparation for Prophet ===")
print(f"Baseline period: {BASELINE_START.date()} → {BASELINE_END.date()}")
print(f"Current period:  Jan 2026 → {SCORING_END.date()}")
print(f"Approach:        Fixed 5Y baseline average comparison")

StatementMeta(, 9969517f-4145-4232-b5a7-9da711710490, 3, Finished, Available, Finished, False)

=== Notebook 1 (Rebuilt): Data Preparation for Prophet ===
Baseline period: 2021-01-01 → 2025-12-01
Current period:  Jan 2026 → 2026-07-01
Approach:        Fixed 5Y baseline average comparison


In [2]:
# ══════════════════════════════════════════════════════════════════
# STEP 1: LOAD ALL RAW SOURCE TABLES
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 1: Loading raw source tables ===")

df_wasde  = load_table("features_wasde")
df_demand = load_table("features_demand", drop_strings=True)
df_ksa    = load_table("features_ksa",    drop_strings=True)
df_bdi    = load_table("features_bdi")
df_policy = load_table("features_policy", drop_strings=True)

print(f"WASDE:   {df_wasde.shape}  | {df_wasde['year_month'].min().date()} → {df_wasde['year_month'].max().date()}")
print(f"Demand:  {df_demand.shape} | {df_demand['year_month'].min().date()} → {df_demand['year_month'].max().date()}")
print(f"KSA:     {df_ksa.shape}    | {df_ksa['year_month'].min().date()} → {df_ksa['year_month'].max().date()}")
print(f"BDI:     {df_bdi.shape}    | {df_bdi['year_month'].min().date()} → {df_bdi['year_month'].max().date()}")
print(f"Policy:  {df_policy.shape} | {df_policy['year_month'].min().date()} → {df_policy['year_month'].max().date()}")

StatementMeta(, 9969517f-4145-4232-b5a7-9da711710490, 4, Finished, Available, Finished, False)


=== Step 1: Loading raw source tables ===
WASDE:   (395, 21)  | 2020-01-01 → 2026-07-01
Demand:  (289, 56) | 2021-01-01 → 2026-06-01
KSA:     (360, 15)    | 2020-01-01 → 2025-12-01
BDI:     (92, 18)    | 2019-01-01 → 2026-08-01
Policy:  (360, 21) | 2020-01-01 → 2025-12-01


In [3]:
# dc = df_wasde[df_wasde["commodity"]=="Rice"].copy()
# dc["year_month"] = pd.to_datetime(dc["year_month"])
# dc = dc.sort_values("year_month")

# print("Rice STU — last 8 months:")
# print(dc.tail(8)[["year_month","stu_ratio","stu_mom_change","stu_3m_trend" if "stu_3m_trend" in dc.columns else "stu_3m_avg"]].to_string(index=False))

# print("\nRice PPI — last 8 months:")
# print(dc.tail(8)[["year_month","ppi_base","ppi_3m_trend"]].to_string(index=False))

# print("\nAugust values historically — PPI (check for real seasonal pattern):")
# aug_values = dc[dc["year_month"].dt.month == 8]
# print(aug_values[["year_month","ppi_base"]].to_string(index=False))

# print("\nAugust values historically — STU:")
# print(aug_values[["year_month","stu_ratio"]].to_string(index=False))

StatementMeta(, 9969517f-4145-4232-b5a7-9da711710490, 5, Finished, Available, Finished, False)

In [4]:
for c in COMMODITIES:
    dc = df_demand[df_demand["commodity"]==c].tail(3)
    print(f"\n{c}:")
    print(dc[["year_month","driver_total_mt"]].to_string(index=False))

StatementMeta(, 9969517f-4145-4232-b5a7-9da711710490, 6, Finished, Available, Finished, False)


Wheat:
year_month  driver_total_mt
2024-10-01     5.774324e+06
2024-11-01     4.697174e+06
2024-12-01     4.878290e+06

Corn:
year_month  driver_total_mt
2023-05-01     6.468568e+06
2023-06-01     6.340999e+06
2023-07-01     6.765545e+06

Rice:
year_month  driver_total_mt
2025-01-01     9.807926e+05
2025-02-01     1.015196e+06
2025-03-01     1.091654e+06

Soybean:
year_month  driver_total_mt
2025-08-01     1.851165e+07
2025-09-01     1.578659e+07
2025-10-01     1.179894e+07

Barley:
year_month  driver_total_mt
2026-04-01     2.934964e+06
2026-05-01     1.449477e+06
2026-06-01     3.200000e-02


In [5]:
# ══════════════════════════════════════════════════════════════════
# STEP 2: STU RATIO TIME SERIES
# Raw:       stu_ratio (monthly %)
# Baseline:  mean of stu_ratio Jan2021-Dec2025 per commodity
# Signal:    deviation = current - baseline_avg
# Direction: negative = below average = risk
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 2: STU ratio — deviation from 5Y baseline ===")

stu_rows     = []
stu_baseline = {}

for commodity in COMMODITIES:
    dc = df_wasde[df_wasde["commodity"] == commodity].copy()
    dc = dc.sort_values("year_month").reset_index(drop=True)
    dc["year_month"] = pd.to_datetime(dc["year_month"])

    # Fill any gaps in time series
    full_range = pd.date_range(dc["year_month"].min(),
                               dc["year_month"].max(), freq="MS")
    dc = dc.set_index("year_month").reindex(full_range)
    dc.index.name = "year_month"
    dc["commodity"] = commodity
    dc["stu_ratio"]  = dc["stu_ratio"].ffill()
    dc = dc.reset_index()

    # Compute fixed 5Y baseline average
    baseline_mask = (
        (dc["year_month"] >= BASELINE_START) &
        (dc["year_month"] <= BASELINE_END)
    )
    baseline_avg = dc.loc[baseline_mask, "stu_ratio"].mean()
    stu_baseline[commodity] = round(baseline_avg, 4)

    print(f"\n{commodity}:")
    print(f"  5Y baseline avg (Jan2021-Dec2025): {baseline_avg:.2f}%")
    print(f"  Latest value:                      {dc['stu_ratio'].iloc[-1]:.2f}%")
    print(f"  Deviation from baseline:           {dc['stu_ratio'].iloc[-1] - baseline_avg:.2f}%")

    # Compute deviation for each month
    dc["baseline_avg"] = baseline_avg
    dc["y"]            = dc["stu_ratio"] - baseline_avg  # deviation
    dc["y_raw"]        = dc["stu_ratio"]

    for _, row in dc.iterrows():
        if pd.notna(row["y"]):
            stu_rows.append({
                "ds":           row["year_month"],
                "y":            round(row["y"], 4),       # deviation from baseline
                "y_raw":        round(row["y_raw"], 4),   # actual STU value
                "baseline_avg": round(baseline_avg, 4),
                "commodity":    commodity,
                "indicator":    "stu_deviation",
                "data_period":  "baseline" if row["year_month"] <= BASELINE_END else "current"
            })

df_stu_ts = pd.DataFrame(stu_rows)
df_stu_ts = df_stu_ts.sort_values(["commodity","ds"]).reset_index(drop=True)

print(f"\nSTU deviation time series: {df_stu_ts.shape}")
print(f"\nDeviation stats (positive = above avg = good, negative = below avg = risk):")
print(df_stu_ts.groupby("commodity")["y"].describe().round(3))

save_to_lakehouse(df_stu_ts, "prophet_ts_stu")

StatementMeta(, 9969517f-4145-4232-b5a7-9da711710490, 7, Finished, Available, Finished, False)


=== Step 2: STU ratio — deviation from 5Y baseline ===

Wheat:
  5Y baseline avg (Jan2021-Dec2025): 26.86%
  Latest value:                      26.25%
  Deviation from baseline:           -0.61%

Corn:
  5Y baseline avg (Jan2021-Dec2025): 21.42%
  Latest value:                      17.99%
  Deviation from baseline:           -3.43%

Rice:
  5Y baseline avg (Jan2021-Dec2025): 30.84%
  Latest value:                      31.80%
  Deviation from baseline:           0.96%

Soybean:
  5Y baseline avg (Jan2021-Dec2025): 19.63%
  Latest value:                      19.64%
  Deviation from baseline:           0.01%

Barley:
  5Y baseline avg (Jan2021-Dec2025): 12.15%
  Latest value:                      13.46%
  Deviation from baseline:           1.31%

STU deviation time series: (347, 7)

Deviation stats (positive = above avg = good, negative = below avg = risk):
           count   mean    std    min    25%    50%    75%    max
commodity                                                        


In [6]:
# # ══════════════════════════════════════════════════════════════════
# # STEP 3: BDI TIME SERIES
# # Raw:       bdi (monthly average index value)
# # Baseline:  mean of bdi Jan2021-Dec2025
# # Signal:    ratio = current / baseline_avg
# # Direction: ratio > 1 = freight elevated = risk
# # ══════════════════════════════════════════════════════════════════

# print("\n=== Step 3: BDI — ratio vs 5Y baseline ===")

# df_bdi_clean = df_bdi.sort_values("year_month").reset_index(drop=True)
# df_bdi_clean["year_month"] = pd.to_datetime(df_bdi_clean["year_month"])

# # Compute fixed 5Y baseline average
# bdi_baseline_mask = (
#     (df_bdi_clean["year_month"] >= BASELINE_START) &
#     (df_bdi_clean["year_month"] <= BASELINE_END)
# )
# bdi_baseline_avg = df_bdi_clean.loc[bdi_baseline_mask, "bdi"].mean()

# print(f"BDI 5Y baseline avg (Jan2021-Dec2025): {bdi_baseline_avg:.0f}")
# print(f"BDI latest value (Jul 2026):           {df_bdi_clean['bdi'].iloc[-1]:.0f}")
# print(f"BDI ratio vs baseline:                 {df_bdi_clean['bdi'].iloc[-1]/bdi_baseline_avg:.3f}")
# print(f"Interpretation:                        {((df_bdi_clean['bdi'].iloc[-1]/bdi_baseline_avg-1)*100):.0f}% above 5Y average")

# # Compute ratio for each month
# df_bdi_clean["baseline_avg"] = bdi_baseline_avg
# df_bdi_clean["y"]            = df_bdi_clean["bdi"] / bdi_baseline_avg  # ratio
# df_bdi_clean["y_raw"]        = df_bdi_clean["bdi"]

# bdi_rows = []
# for commodity in COMMODITIES:
#     for _, row in df_bdi_clean.iterrows():
#         bdi_rows.append({
#             "ds":           row["year_month"],
#             "y":            round(row["y"], 4),       # ratio vs baseline
#             "y_raw":        round(row["y_raw"], 2),   # actual BDI
#             "baseline_avg": round(bdi_baseline_avg, 2),
#             "commodity":    commodity,
#             "indicator":    "bdi_ratio",
#             "data_period":  "baseline" if row["year_month"] <= BASELINE_END else "current"
#         })

# df_bdi_ts = pd.DataFrame(bdi_rows).dropna(subset=["y"])
# df_bdi_ts = df_bdi_ts.sort_values(["commodity","ds"]).reset_index(drop=True)

# print(f"\nBDI ratio time series: {df_bdi_ts.shape}")
# print(f"\nBDI ratio stats (1.0 = at baseline, >1 = elevated):")
# print(df_bdi_ts[df_bdi_ts["commodity"]=="Wheat"]["y"].describe().round(3))

# # save_to_lakehouse(df_bdi_ts, "prophet_ts_bdi")

StatementMeta(, 9969517f-4145-4232-b5a7-9da711710490, 8, Finished, Available, Finished, False)

In [7]:
# print("\n=== Step 3: BDI — level + momentum vs pre-crisis baseline ===")

# df_bdi_clean = df_bdi.sort_values("year_month").reset_index(drop=True)
# df_bdi_clean["year_month"] = pd.to_datetime(df_bdi_clean["year_month"])

# # ── Component 1: LEVEL — median over pre-crisis window ──────────
# # Houthi attacks on Red Sea shipping began Nov 2023 — excluding that
# # entirely (rather than trying to statistically dampen it) is the only
# # way to remove a sustained, ongoing disruption from the baseline.
# # Within this window, COVID (2020 crash + 2021 surge to 5,649) is a
# # genuine outlier pair, so median (not mean) is used to avoid distortion.
# BDI_BASELINE_START = pd.Timestamp("2019-01-01")
# BDI_BASELINE_END   = pd.Timestamp("2023-10-01")

# bdi_baseline_mask = (
#     (df_bdi_clean["year_month"] >= BDI_BASELINE_START) &
#     (df_bdi_clean["year_month"] <= BDI_BASELINE_END)
# )
# bdi_baseline_median = df_bdi_clean.loc[bdi_baseline_mask, "bdi"].median()
# bdi_baseline_n      = bdi_baseline_mask.sum()

# df_bdi_clean["level_ratio"] = df_bdi_clean["bdi"] / bdi_baseline_median

# # ── Component 2: MOMENTUM — current vs trailing 6-month average ─
# # Captures active escalation, independent of long-run history.
# # min_periods=3 allows the ratio to compute even early in the series
# # rather than requiring a full 6 months before producing a value.
# df_bdi_clean["rolling_6m_avg"] = (
#     df_bdi_clean["bdi"].rolling(window=6, min_periods=3).mean()
# )
# df_bdi_clean["momentum_ratio"] = df_bdi_clean["bdi"] / df_bdi_clean["rolling_6m_avg"]

# # ── Combined signal — 60% level, 40% momentum ────────────────────
# LEVEL_WEIGHT    = 0.60
# MOMENTUM_WEIGHT = 0.40

# df_bdi_clean["combined_ratio"] = (
#     LEVEL_WEIGHT * df_bdi_clean["level_ratio"] +
#     MOMENTUM_WEIGHT * df_bdi_clean["momentum_ratio"]
# )

# # ── Reporting ─────────────────────────────────────────────────────
# latest = df_bdi_clean.iloc[-1]
# print(f"BDI pre-crisis baseline median ({BDI_BASELINE_START.strftime('%b %Y')}-{BDI_BASELINE_END.strftime('%b %Y')}, n={bdi_baseline_n}): {bdi_baseline_median:.0f}")
# print(f"BDI latest value ({latest['year_month'].strftime('%b %Y')}):           {latest['bdi']:.0f}")
# print(f"  Level ratio (vs pre-crisis median):    {latest['level_ratio']:.3f}  ({(latest['level_ratio']-1)*100:+.0f}% vs pre-crisis)")
# print(f"  Momentum ratio (vs trailing 6M avg):    {latest['momentum_ratio']:.3f}  ({(latest['momentum_ratio']-1)*100:+.0f}% vs recent trend)")
# print(f"  Combined signal ({LEVEL_WEIGHT:.0%} level + {MOMENTUM_WEIGHT:.0%} momentum): {latest['combined_ratio']:.3f}")

# # ── Build output rows — same shape the rest of the pipeline expects ─
# df_bdi_clean["y"]            = df_bdi_clean["combined_ratio"]
# df_bdi_clean["y_raw"]        = df_bdi_clean["bdi"]
# df_bdi_clean["baseline_avg"] = bdi_baseline_median  # kept for reference/transparency

# bdi_rows = []
# for commodity in COMMODITIES:
#     for _, row in df_bdi_clean.iterrows():
#         if pd.notna(row["y"]):
#             bdi_rows.append({
#                 "ds":              row["year_month"],
#                 "y":               round(row["y"], 4),
#                 "y_raw":           round(row["y_raw"], 2),
#                 "baseline_avg":    round(bdi_baseline_median, 2),
#                 "level_ratio":     round(row["level_ratio"], 4),
#                 "momentum_ratio":  round(row["momentum_ratio"], 4) if pd.notna(row["momentum_ratio"]) else None,
#                 "commodity":       commodity,
#                 "indicator":       "bdi_ratio",
#                 "data_period":     "baseline" if row["year_month"] <= BASELINE_END else "current"
#             })

# df_bdi_ts = pd.DataFrame(bdi_rows)
# df_bdi_ts = df_bdi_ts.sort_values(["commodity","ds"]).reset_index(drop=True)

# print(f"\nBDI combined signal time series: {df_bdi_ts.shape}")
# print(f"\nCombined signal stats (1.0 = at pre-crisis baseline & flat trend):")
# print(df_bdi_ts[df_bdi_ts["commodity"]=="Wheat"]["y"].describe().round(3))

# save_to_lakehouse(df_bdi_ts, "prophet_ts_bdi")

StatementMeta(, 9969517f-4145-4232-b5a7-9da711710490, 9, Finished, Available, Finished, False)

In [8]:
print("\n=== Step 3: BDI — level + momentum vs pre-crisis baseline ===")

df_bdi_clean = df_bdi.sort_values("year_month").reset_index(drop=True)
df_bdi_clean["year_month"] = pd.to_datetime(df_bdi_clean["year_month"])

# ── Component 1: LEVEL — median over pre-crisis window ──────────
# Houthi attacks on Red Sea shipping began Nov 2023 — excluding that
# entirely (rather than trying to statistically dampen it) is the only
# way to remove a sustained, ongoing disruption from the baseline.
# Within this window, COVID (2020 crash + 2021 surge to 5,649) is a
# genuine outlier pair, so median (not mean) is used to avoid distortion.
BDI_BASELINE_START = pd.Timestamp("2019-01-01")
BDI_BASELINE_END   = pd.Timestamp("2023-10-01")

bdi_baseline_mask = (
    (df_bdi_clean["year_month"] >= BDI_BASELINE_START) &
    (df_bdi_clean["year_month"] <= BDI_BASELINE_END)
)
bdi_baseline_median = df_bdi_clean.loc[bdi_baseline_mask, "bdi"].median()
bdi_baseline_n      = bdi_baseline_mask.sum()

df_bdi_clean["level_ratio"] = df_bdi_clean["bdi"] / bdi_baseline_median

# ── Component 2: MOMENTUM (6M) — current vs trailing 6-month average ─
# Smoother, more stable medium-term trend signal.
df_bdi_clean["rolling_6m_avg"] = (
    df_bdi_clean["bdi"].rolling(window=6, min_periods=3).mean()
)
df_bdi_clean["momentum_6m_ratio"] = df_bdi_clean["bdi"] / df_bdi_clean["rolling_6m_avg"]

# ── Component 3: MOMENTUM (3M) — current vs trailing 3-month average ─
# More responsive to acute, fresh escalation (e.g. a new chokepoint
# event layered on top of an already-ongoing crisis). Noisier than 6M,
# so it gets more weight than 6M but doesn't dominate the level component.
df_bdi_clean["rolling_3m_avg"] = (
    df_bdi_clean["bdi"].rolling(window=3, min_periods=2).mean()
)
df_bdi_clean["momentum_3m_ratio"] = df_bdi_clean["bdi"] / df_bdi_clean["rolling_3m_avg"]

# ── Combined signal — 60% level, 25% 3M momentum, 15% 6M momentum ─
LEVEL_WEIGHT        = 0.60
MOMENTUM_3M_WEIGHT  = 0.25
MOMENTUM_6M_WEIGHT  = 0.15

df_bdi_clean["combined_ratio"] = (
    LEVEL_WEIGHT * df_bdi_clean["level_ratio"] +
    MOMENTUM_3M_WEIGHT * df_bdi_clean["momentum_3m_ratio"] +
    MOMENTUM_6M_WEIGHT * df_bdi_clean["momentum_6m_ratio"]
)

# ── Reporting ─────────────────────────────────────────────────────
latest = df_bdi_clean.iloc[-1]
print(f"BDI pre-crisis baseline median ({BDI_BASELINE_START.strftime('%b %Y')}-{BDI_BASELINE_END.strftime('%b %Y')}, n={bdi_baseline_n}): {bdi_baseline_median:.0f}")
print(f"BDI latest value ({latest['year_month'].strftime('%b %Y')}):           {latest['bdi']:.0f}")
print(f"  Level ratio (vs pre-crisis median):     {latest['level_ratio']:.3f}  ({(latest['level_ratio']-1)*100:+.0f}% vs pre-crisis)")
print(f"  3M momentum ratio (vs trailing 3M avg):  {latest['momentum_3m_ratio']:.3f}  ({(latest['momentum_3m_ratio']-1)*100:+.0f}% vs recent 3M trend)")
print(f"  6M momentum ratio (vs trailing 6M avg):  {latest['momentum_6m_ratio']:.3f}  ({(latest['momentum_6m_ratio']-1)*100:+.0f}% vs recent 6M trend)")
print(f"  Combined signal ({LEVEL_WEIGHT:.0%} level + {MOMENTUM_3M_WEIGHT:.0%} 3M + {MOMENTUM_6M_WEIGHT:.0%} 6M): {latest['combined_ratio']:.3f}")

# ── Build output rows — same shape the rest of the pipeline expects ─
df_bdi_clean["y"]            = df_bdi_clean["combined_ratio"]
df_bdi_clean["y_raw"]        = df_bdi_clean["bdi"]
df_bdi_clean["baseline_avg"] = bdi_baseline_median  # kept for reference/transparency

bdi_rows = []
for commodity in COMMODITIES:
    for _, row in df_bdi_clean.iterrows():
        if pd.notna(row["y"]):
            bdi_rows.append({
                "ds":                row["year_month"],
                "y":                 round(row["y"], 4),
                "y_raw":             round(row["y_raw"], 2),
                "baseline_avg":      round(bdi_baseline_median, 2),
                "level_ratio":       round(row["level_ratio"], 4),
                "momentum_3m_ratio": round(row["momentum_3m_ratio"], 4) if pd.notna(row["momentum_3m_ratio"]) else None,
                "momentum_6m_ratio": round(row["momentum_6m_ratio"], 4) if pd.notna(row["momentum_6m_ratio"]) else None,
                "commodity":         commodity,
                "indicator":         "bdi_ratio",
                "data_period":       "baseline" if row["year_month"] <= BASELINE_END else "current"
            })

df_bdi_ts = pd.DataFrame(bdi_rows)
df_bdi_ts = df_bdi_ts.sort_values(["commodity","ds"]).reset_index(drop=True)

print(f"\nBDI combined signal time series: {df_bdi_ts.shape}")
print(f"\nCombined signal stats (1.0 = at pre-crisis baseline & flat trend):")
print(df_bdi_ts[df_bdi_ts["commodity"]=="Wheat"]["y"].describe().round(3))

save_to_lakehouse(df_bdi_ts, "prophet_ts_bdi")

StatementMeta(, 9969517f-4145-4232-b5a7-9da711710490, 10, Finished, Available, Finished, False)


=== Step 3: BDI — level + momentum vs pre-crisis baseline ===
BDI pre-crisis baseline median (Jan 2019-Oct 2023, n=58): 1484
BDI latest value (Aug 2026):           2976
  Level ratio (vs pre-crisis median):     2.006  (+101% vs pre-crisis)
  3M momentum ratio (vs trailing 3M avg):  1.048  (+5% vs recent 3M trend)
  6M momentum ratio (vs trailing 6M avg):  1.112  (+11% vs recent 6M trend)
  Combined signal (60% level + 25% 3M + 15% 6M): 1.632

BDI combined signal time series: (450, 10)

Combined signal stats (1.0 = at pre-crisis baseline & flat trend):
count    90.000
mean      1.146
std       0.364
min       0.374
25%       0.937
50%       1.121
75%       1.333
max       2.429
Name: y, dtype: float64
✓ srm.prophet_ts_bdi: 450 rows saved


In [9]:
# ══════════════════════════════════════════════════════════════════
# STEP 4: DEMAND PRESSURE TIME SERIES
# Raw:       driver_total_mt (total import volume MT
#            from all major tracked buyers per commodity)
# Baseline:  mean of driver_total_mt Jan2021-Dec2025
# Signal:    ratio = current / baseline_avg
# Direction: ratio > 1 = demand elevated = risk
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 4: Demand — ratio vs 5Y baseline ===")

demand_rows     = []
demand_baseline = {}

for commodity in COMMODITIES:
    dc = df_demand[df_demand["commodity"] == commodity].copy()
    dc = dc.sort_values("year_month").reset_index(drop=True)
    dc["year_month"] = pd.to_datetime(dc["year_month"])

    # Fill any month gaps with 0 (no data = no tracked buyer activity)
    full_range = pd.date_range(dc["year_month"].min(),
                               dc["year_month"].max(), freq="MS")
    dc = dc.set_index("year_month").reindex(full_range)
    dc.index.name = "year_month"
    dc["commodity"]      = commodity
    dc["driver_total_mt"] = dc["driver_total_mt"].fillna(0)
    dc = dc.reset_index()

    # Compute fixed 5Y baseline average
    baseline_mask = (
        (dc["year_month"] >= BASELINE_START) &
        (dc["year_month"] <= BASELINE_END)
    )
    baseline_avg = dc.loc[baseline_mask, "driver_total_mt"].mean()
    demand_baseline[commodity] = round(baseline_avg, 2)

    print(f"\n{commodity}:")
    print(f"  5Y baseline avg imports: {baseline_avg/1e6:.2f}M MT/month")
    print(f"  Latest value:            {dc['driver_total_mt'].iloc[-1]/1e6:.2f}M MT")

    # Compute ratio — handle division by zero
    if baseline_avg > 0:
        dc["y"] = dc["driver_total_mt"] / baseline_avg
    else:
        dc["y"] = 1.0

    dc["y_raw"]        = dc["driver_total_mt"]
    dc["baseline_avg"] = baseline_avg

    print(f"  Latest ratio vs baseline: {dc['y'].iloc[-1]:.3f}")

    for _, row in dc.iterrows():
        demand_rows.append({
            "ds":           row["year_month"],
            "y":            round(float(row["y"]), 4),
            "y_raw":        round(float(row["y_raw"]), 2),
            "baseline_avg": round(baseline_avg, 2),
            "commodity":    commodity,
            "indicator":    "demand_ratio",
            "data_period":  "baseline" if row["year_month"] <= BASELINE_END else "current"
        })

df_demand_ts = pd.DataFrame(demand_rows)
df_demand_ts = df_demand_ts.sort_values(["commodity","ds"]).reset_index(drop=True)

print(f"\nDemand ratio time series: {df_demand_ts.shape}")
print(f"\nDemand ratio stats (1.0 = at baseline, >1 = elevated):")
print(df_demand_ts.groupby("commodity")["y"].describe().round(3))

save_to_lakehouse(df_demand_ts, "prophet_ts_demand")

StatementMeta(, 9969517f-4145-4232-b5a7-9da711710490, 11, Finished, Available, Finished, False)


=== Step 4: Demand — ratio vs 5Y baseline ===

Wheat:
  5Y baseline avg imports: 5.97M MT/month
  Latest value:            0.59M MT
  Latest ratio vs baseline: 0.099

Corn:
  5Y baseline avg imports: 6.98M MT/month
  Latest value:            0.19M MT
  Latest ratio vs baseline: 0.027

Rice:
  5Y baseline avg imports: 1.26M MT/month
  Latest value:            0.11M MT
  Latest ratio vs baseline: 0.085

Soybean:
  5Y baseline avg imports: 27.59M MT/month
  Latest value:            21.47M MT
  Latest ratio vs baseline: 0.778

Barley:
  5Y baseline avg imports: 3.61M MT/month
  Latest value:            0.00M MT
  Latest ratio vs baseline: 0.000

Demand ratio time series: (289, 7)

Demand ratio stats (1.0 = at baseline, >1 = elevated):
           count   mean    std    min    25%    50%    75%    max
commodity                                                        
Barley      66.0  0.969  0.284  0.000  0.808  0.973  1.110  1.706
Corn        52.0  0.961  0.278  0.027  0.817  0.958  1.138  

In [10]:
# ══════════════════════════════════════════════════════════════════
# STEP 5: PPI TIME SERIES
# Raw:       ppi_base from features_wasde
#            ppi_base = (production / 12m_rolling_avg) × 100
#            value of 100 = production at 12m average
# Baseline:  mean of ppi_base Jan2021-Dec2025 per commodity
# Signal:    ratio = current / baseline_avg
# Direction: ratio < 1 = production below baseline = risk
# Note:      ppi_base already normalised around 100
#            baseline_avg will be close to 100
#            ratio will be close to 1
#            Deviation = current - baseline is more meaningful
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 5: PPI — deviation vs 5Y baseline ===")

ppi_rows     = []
ppi_baseline = {}

for commodity in COMMODITIES:
    dc = df_wasde[df_wasde["commodity"] == commodity].copy()
    dc = dc.sort_values("year_month").reset_index(drop=True)
    dc["year_month"] = pd.to_datetime(dc["year_month"])
    dc["ppi_base"]   = dc["ppi_base"].ffill()

    # Compute fixed 5Y baseline average
    baseline_mask = (
        (dc["year_month"] >= BASELINE_START) &
        (dc["year_month"] <= BASELINE_END)
    )
    baseline_avg = dc.loc[baseline_mask, "ppi_base"].mean()
    ppi_baseline[commodity] = round(baseline_avg, 4)

    print(f"\n{commodity}:")
    print(f"  5Y baseline avg PPI:   {baseline_avg:.2f}")
    print(f"  Latest PPI value:      {dc['ppi_base'].iloc[-1]:.2f}")
    print(f"  Deviation:             {dc['ppi_base'].iloc[-1] - baseline_avg:.2f}")
    print(f"  Interpretation:        production {'above' if dc['ppi_base'].iloc[-1] >= baseline_avg else 'below'} 5Y average")

    # Use deviation = current - baseline
    # Positive = production above baseline = good
    # Negative = production below baseline = risk
    dc["baseline_avg"] = baseline_avg
    dc["y"]            = dc["ppi_base"] - baseline_avg  # deviation
    dc["y_raw"]        = dc["ppi_base"]

    for _, row in dc.iterrows():
        if pd.notna(row["y"]):
            ppi_rows.append({
                "ds":           row["year_month"],
                "y":            round(float(row["y"]), 4),
                "y_raw":        round(float(row["y_raw"]), 4),
                "baseline_avg": round(baseline_avg, 4),
                "commodity":    commodity,
                "indicator":    "ppi_deviation",
                "data_period":  "baseline" if row["year_month"] <= BASELINE_END else "current"
            })

df_ppi_ts = pd.DataFrame(ppi_rows)
df_ppi_ts = df_ppi_ts.sort_values(["commodity","ds"]).reset_index(drop=True)

print(f"\nPPI deviation time series: {df_ppi_ts.shape}")
print(f"\nPPI deviation stats (positive = above avg = good):")
print(df_ppi_ts.groupby("commodity")["y"].describe().round(3))

save_to_lakehouse(df_ppi_ts, "prophet_ts_ppi")

StatementMeta(, 9969517f-4145-4232-b5a7-9da711710490, 12, Finished, Available, Finished, False)


=== Step 5: PPI — deviation vs 5Y baseline ===

Wheat:
  5Y baseline avg PPI:   100.62
  Latest PPI value:      99.23
  Deviation:             -1.39
  Interpretation:        production below 5Y average

Corn:
  5Y baseline avg PPI:   101.01
  Latest PPI value:      100.68
  Deviation:             -0.33
  Interpretation:        production below 5Y average

Rice:
  5Y baseline avg PPI:   100.86
  Latest PPI value:      99.36
  Deviation:             -1.50
  Interpretation:        production below 5Y average

Soybean:
  5Y baseline avg PPI:   101.51
  Latest PPI value:      103.07
  Deviation:             1.56
  Interpretation:        production above 5Y average

Barley:
  5Y baseline avg PPI:   99.42
  Latest PPI value:      103.05
  Deviation:             3.63
  Interpretation:        production above 5Y average

PPI deviation time series: (287, 7)

PPI deviation stats (positive = above avg = good):
           count   mean    std    min    25%    50%    75%    max
commodity            

In [11]:
# ══════════════════════════════════════════════════════════════════
# STEP 6: KSA CONCENTRATION TIME SERIES
# Raw:       ksa_top3_share (decimal e.g. 0.97 = 97%)
#            Annual data — same value repeated each month
# Baseline:  mean of ksa_top3_share Jan2021-Dec2025 per commodity
# Signal:    deviation = current% - baseline_avg%
# Direction: positive = more concentrated than avg = risk
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 6: KSA concentration — deviation vs 5Y baseline ===")

ksa_rows     = []
ksa_baseline = {}

for commodity in COMMODITIES:
    dc = df_ksa[df_ksa["commodity"] == commodity].copy()
    dc = dc.sort_values("year_month").reset_index(drop=True)
    dc["year_month"] = pd.to_datetime(dc["year_month"])

    # Convert to percentage
    dc["ksa_top3_pct"] = dc["ksa_top3_share"] * 100
    dc["ksa_top3_pct"] = dc["ksa_top3_pct"].ffill()

    # Fill any month gaps
    full_range = pd.date_range(dc["year_month"].min(),
                               dc["year_month"].max(), freq="MS")
    dc = dc.set_index("year_month").reindex(full_range)
    dc.index.name = "year_month"
    dc["commodity"]    = commodity
    dc["ksa_top3_pct"] = dc["ksa_top3_pct"].ffill()
    dc = dc.reset_index()

    # Compute fixed 5Y baseline average
    baseline_mask = (
        (dc["year_month"] >= BASELINE_START) &
        (dc["year_month"] <= BASELINE_END)
    )
    baseline_avg = dc.loc[baseline_mask, "ksa_top3_pct"].mean()
    ksa_baseline[commodity] = round(baseline_avg, 4)

    print(f"\n{commodity}:")
    print(f"  5Y baseline avg top3 share: {baseline_avg:.1f}%")
    print(f"  Latest value:               {dc['ksa_top3_pct'].iloc[-1]:.1f}%")
    print(f"  Deviation from baseline:    {dc['ksa_top3_pct'].iloc[-1] - baseline_avg:+.1f}%")
    print(f"  Interpretation:             concentration {'rising above' if dc['ksa_top3_pct'].iloc[-1] > baseline_avg else 'below'} 5Y average")

    # Deviation = current - baseline
    dc["baseline_avg"] = baseline_avg
    dc["y"]            = dc["ksa_top3_pct"] - baseline_avg  # deviation in %
    dc["y_raw"]        = dc["ksa_top3_pct"]

    for _, row in dc.iterrows():
        if pd.notna(row["y"]):
            ksa_rows.append({
                "ds":           row["year_month"],
                "y":            round(float(row["y"]), 4),
                "y_raw":        round(float(row["y_raw"]), 4),
                "baseline_avg": round(baseline_avg, 4),
                "commodity":    commodity,
                "indicator":    "ksa_deviation",
                "data_period":  "baseline" if row["year_month"] <= BASELINE_END else "current"
            })

df_ksa_ts = pd.DataFrame(ksa_rows)
df_ksa_ts = df_ksa_ts.sort_values(["commodity","ds"]).reset_index(drop=True)

print(f"\nKSA deviation time series: {df_ksa_ts.shape}")
print(f"\nKSA deviation stats (positive = more concentrated than baseline = risk):")
print(df_ksa_ts.groupby("commodity")["y"].describe().round(3))

save_to_lakehouse(df_ksa_ts, "prophet_ts_ksa")

StatementMeta(, 9969517f-4145-4232-b5a7-9da711710490, 13, Finished, Available, Finished, False)


=== Step 6: KSA concentration — deviation vs 5Y baseline ===

Wheat:
  5Y baseline avg top3 share: 81.9%
  Latest value:               87.1%
  Deviation from baseline:    +5.2%
  Interpretation:             concentration rising above 5Y average

Corn:
  5Y baseline avg top3 share: 96.8%
  Latest value:               97.4%
  Deviation from baseline:    +0.6%
  Interpretation:             concentration rising above 5Y average

Rice:
  5Y baseline avg top3 share: 92.4%
  Latest value:               93.1%
  Deviation from baseline:    +0.8%
  Interpretation:             concentration rising above 5Y average

Soybean:
  5Y baseline avg top3 share: 98.6%
  Latest value:               100.0%
  Deviation from baseline:    +1.4%
  Interpretation:             concentration rising above 5Y average

Barley:
  5Y baseline avg top3 share: 82.0%
  Latest value:               56.1%
  Deviation from baseline:    -25.9%
  Interpretation:             concentration below 5Y average

KSA deviation time se

In [12]:
# df_master = spark.table("srm.features_master").toPandas()
# df_master["year_month"] = pd.to_datetime(df_master["year_month"])

# # Get the last row where country data actually exists, not just the latest month
# dc_real = df_master[df_master["ksa_top1_country"].notna()].sort_values("year_month").groupby("commodity").tail(1)
# print(dc_real[["commodity","year_month","ksa_top1_country","ksa_top3_countries","ksa_top1_share","ksa_top3_share"]].to_string(index=False))


StatementMeta(, 9969517f-4145-4232-b5a7-9da711710490, 14, Finished, Available, Finished, False)

In [13]:
# # ══════════════════════════════════════════════════════════════════
# # STEP X: FOB PRICE TIME SERIES
# # Raw:       price (monthly, USD/mt or EUR/mt, last obs of month)
# # Baseline:  mean of price Jan2021-Dec2025 per commodity
# # Signal:    deviation = current - baseline_avg  (raw currency units)
# # Direction: positive = price above average = risk (higher import cost)
# # ══════════════════════════════════════════════════════════════════

# print("\n=== Step X: FOB Price — deviation from 5Y baseline ===")

# df_price_raw   = spark.table("srm.features_price").toPandas()
# price_rows     = []
# price_baseline = {}

# for commodity in COMMODITIES:
#     dc = df_price_raw[df_price_raw["commodity"] == commodity].copy()
#     dc = dc.sort_values("year_month").reset_index(drop=True)
#     dc["year_month"] = pd.to_datetime(dc["year_month"])

#     full_range = pd.date_range(dc["year_month"].min(), dc["year_month"].max(), freq="MS")
#     dc = dc.set_index("year_month").reindex(full_range)
#     dc.index.name = "year_month"
#     dc["commodity"] = commodity
#     dc["price"] = dc["price"].ffill()
#     dc = dc.reset_index()

#     baseline_mask = (dc["year_month"] >= BASELINE_START) & (dc["year_month"] <= BASELINE_END)
#     baseline_avg = dc.loc[baseline_mask, "price"].mean()
#     price_baseline[commodity] = round(baseline_avg, 4)

#     print(f"\n{commodity}:")
#     print(f"  5Y baseline avg (Jan2021-Dec2025): {baseline_avg:.2f}")
#     print(f"  Latest value:                      {dc['price'].iloc[-1]:.2f}")
#     print(f"  Deviation from baseline:           {dc['price'].iloc[-1] - baseline_avg:.2f}")

#     dc["baseline_avg"] = baseline_avg
#     dc["y"]     = dc["price"] - baseline_avg
#     dc["y_raw"] = dc["price"]

#     for _, row in dc.iterrows():
#         if pd.notna(row["y"]):
#             price_rows.append({
#                 "ds":           row["year_month"],
#                 "y":            round(row["y"], 4),
#                 "y_raw":        round(row["y_raw"], 4),
#                 "baseline_avg": round(baseline_avg, 4),
#                 "commodity":    commodity,
#                 "indicator":    "fob_price_deviation",
#                 "data_period":  "baseline" if row["year_month"] <= BASELINE_END else "current"
#             })

# df_price_ts = pd.DataFrame(price_rows)
# df_price_ts = df_price_ts.sort_values(["commodity","ds"]).reset_index(drop=True)

# print(f"\nFOB Price deviation time series: {df_price_ts.shape}")
# save_to_lakehouse(df_price_ts, "prophet_ts_price")


print("\n=== Step X: FOB Price — log-price signal for Prophet, baseline stored separately for scoring ===")

df_price_raw   = spark.table("srm.features_price").toPandas()
price_rows     = []
price_baseline = {}

for commodity in COMMODITIES:
    dc = df_price_raw[df_price_raw["commodity"] == commodity].copy()
    dc = dc.sort_values("year_month").reset_index(drop=True)
    dc["year_month"] = pd.to_datetime(dc["year_month"])

    full_range = pd.date_range(dc["year_month"].min(), dc["year_month"].max(), freq="MS")
    dc = dc.set_index("year_month").reindex(full_range)
    dc.index.name = "year_month"
    dc["commodity"] = commodity
    dc["price"] = dc["price"].ffill()
    dc = dc.reset_index()

    baseline_mask = (dc["year_month"] >= BASELINE_START) & (dc["year_month"] <= BASELINE_END)
    baseline_avg = dc.loc[baseline_mask, "price"].mean()   # still real-price average — used only for scoring, not training
    price_baseline[commodity] = round(baseline_avg, 4)

    print(f"\n{commodity}:")
    print(f"  5Y baseline avg (Jan2021-Dec2025): {baseline_avg:.2f}")
    print(f"  Latest value:                      {dc['price'].iloc[-1]:.2f}")
    print(f"  Deviation from baseline:           {dc['price'].iloc[-1] - baseline_avg:.2f}")

    # ── UPDATED (Venkat) — Prophet now trains on log(price) directly,
    # not a pre-subtracted deviation. Prophet's own trend/changepoint
    # detection handles the real price trend, instead of us relabeling
    # the trend by subtracting a fixed 2021-2025 baseline first.
    dc["baseline_avg"] = baseline_avg
    dc["y"]     = np.log(dc["price"])   # NEW — Prophet's actual training target
    dc["y_raw"] = dc["price"]           # unchanged — still the real price, for reference

    for _, row in dc.iterrows():
        if pd.notna(row["y"]):
            price_rows.append({
                "ds":           row["year_month"],
                "y":            round(row["y"], 6),   # more decimal places — log values are small (~5-6 range)
                "y_raw":        round(row["y_raw"], 4),
                "baseline_avg": round(baseline_avg, 4),
                "commodity":    commodity,
                "indicator":    "fob_price_deviation",
                "data_period":  "baseline" if row["year_month"] <= BASELINE_END else "current"
            })

df_price_ts = pd.DataFrame(price_rows)
df_price_ts = df_price_ts.sort_values(["commodity","ds"]).reset_index(drop=True)

print(f"\nFOB Price log-price time series: {df_price_ts.shape}")
save_to_lakehouse(df_price_ts, "prophet_ts_price")

StatementMeta(, 9969517f-4145-4232-b5a7-9da711710490, 15, Finished, Available, Finished, False)


=== Step X: FOB Price — log-price signal for Prophet, baseline stored separately for scoring ===

Wheat:
  5Y baseline avg (Jan2021-Dec2025): 270.77
  Latest value:                      219.00
  Deviation from baseline:           -51.77

Corn:
  5Y baseline avg (Jan2021-Dec2025): 233.82
  Latest value:                      212.25
  Deviation from baseline:           -21.57

Rice:
  5Y baseline avg (Jan2021-Dec2025): 332.87
  Latest value:                      308.26
  Deviation from baseline:           -24.61

Soybean:
  5Y baseline avg (Jan2021-Dec2025): 504.83
  Latest value:                      492.75
  Deviation from baseline:           -12.08

Barley:
  5Y baseline avg (Jan2021-Dec2025): 238.52
  Latest value:                      193.00
  Deviation from baseline:           -45.52

FOB Price log-price time series: (524, 7)
✓ srm.prophet_ts_price: 524 rows saved


In [14]:
print("\n=== Step X: FOB Price — 12-month rolling baseline signal (test candidate) ===")

df_price_raw = spark.table("srm.features_price").toPandas()
rolling_rows = []
frozen_baseline = {}   # last known real rolling baseline per commodity — used to reconstruct forecast months
price_baseline  = {}
for commodity in COMMODITIES:
    dc = df_price_raw[df_price_raw["commodity"] == commodity].copy()
    dc = dc.sort_values("year_month").reset_index(drop=True)
    dc["year_month"] = pd.to_datetime(dc["year_month"])

    full_range = pd.date_range(dc["year_month"].min(), dc["year_month"].max(), freq="MS")
    dc = dc.set_index("year_month").reindex(full_range)
    dc.index.name = "year_month"
    dc["commodity"] = commodity
    dc["price"] = dc["price"].ffill()
    dc = dc.reset_index()

    # ── Trailing 12-month rolling average, computed at every point in history ──
    dc["rolling_baseline"] = dc["price"].rolling(window=12, min_periods=12).mean()

    # Still keep the fixed 5Y baseline too — needed downstream for scoring thresholds,
    # same role it's always played, unrelated to what Prophet trains on.
    baseline_mask = (dc["year_month"] >= BASELINE_START) & (dc["year_month"] <= BASELINE_END)
    fixed_baseline_avg = dc.loc[baseline_mask, "price"].mean()
    price_baseline[commodity] = round(fixed_baseline_avg, 4) 

    # Freeze the rolling baseline at its last known REAL value — used for all 3 forecast months
    last_real_row = dc[dc["price"].notna() & dc["rolling_baseline"].notna()].iloc[-1]
    frozen_baseline[commodity] = round(float(last_real_row["rolling_baseline"]), 4)

    print(f"\n{commodity}:")
    print(f"  Fixed 5Y baseline (still used for scoring): {fixed_baseline_avg:.2f}")
    print(f"  Frozen rolling baseline (as of {last_real_row['year_month'].date()}): {frozen_baseline[commodity]:.2f}")

    dc["y"]     = dc["price"] - dc["rolling_baseline"]   # NEW — deviation from TIME-VARYING rolling baseline
    dc["y_raw"] = dc["price"]

    for _, row in dc.iterrows():
        if pd.notna(row["y"]):   # first 11 months have no rolling baseline yet — excluded automatically
            rolling_rows.append({
                "ds":                row["year_month"],
                "y":                 round(row["y"], 4),
                "y_raw":             round(row["y_raw"], 4),
                "rolling_baseline":  round(row["rolling_baseline"], 4),   # NEW — point-in-time baseline, needed for reconstruction
                "baseline_avg":      round(fixed_baseline_avg, 4),         # kept — still the scoring-time reference
                "frozen_baseline":   frozen_baseline[commodity],           # NEW — used only for forecast-period reconstruction
                "commodity":         commodity,
                "indicator":         "fob_price_deviation_rolling",        # NEW indicator name — kept separate from the log-price version so both can be tested side by side
                "data_period":       "baseline" if row["year_month"] <= BASELINE_END else "current"
            })

df_price_rolling_ts = pd.DataFrame(rolling_rows)
df_price_rolling_ts = df_price_rolling_ts.sort_values(["commodity","ds"]).reset_index(drop=True)

print(f"\nRolling-baseline Price time series: {df_price_rolling_ts.shape}")
save_to_lakehouse(df_price_rolling_ts, "prophet_ts_price_rolling")

StatementMeta(, 9969517f-4145-4232-b5a7-9da711710490, 16, Finished, Available, Finished, False)


=== Step X: FOB Price — 12-month rolling baseline signal (test candidate) ===

Wheat:
  Fixed 5Y baseline (still used for scoring): 270.77
  Frozen rolling baseline (as of 2026-08-01): 232.04

Corn:
  Fixed 5Y baseline (still used for scoring): 233.82
  Frozen rolling baseline (as of 2026-08-01): 208.81

Rice:
  Fixed 5Y baseline (still used for scoring): 332.87
  Frozen rolling baseline (as of 2026-08-01): 253.69

Soybean:
  Fixed 5Y baseline (still used for scoring): 504.83
  Frozen rolling baseline (as of 2026-08-01): 444.88

Barley:
  Fixed 5Y baseline (still used for scoring): 238.52
  Frozen rolling baseline (as of 2026-08-01): 225.67

Rolling-baseline Price time series: (469, 9)
✓ srm.prophet_ts_price_rolling: 469 rows saved


In [15]:
# ══════════════════════════════════════════════════════════════════
# STEP 7: POLICY TIME SERIES
# Rule-based — no 5Y avg applicable
# Carry forward as-is from features_policy
# policy_risk_score_weighted already 0-100
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 7: Policy — rule-based (no 5Y avg) ===")

policy_rows = []

for commodity in COMMODITIES:
    dc = df_policy[df_policy["commodity"] == commodity].copy()
    dc = dc.sort_values("year_month").reset_index(drop=True)
    dc["year_month"] = pd.to_datetime(dc["year_month"])
    dc["policy_risk_score_weighted"] = dc["policy_risk_score_weighted"].fillna(0)

    # No baseline comparison — policy is event-driven
    # Use raw score directly
    for _, row in dc.iterrows():
        policy_rows.append({
            "ds":           row["year_month"],
            "y":            round(float(row["policy_risk_score_weighted"]), 4),
            "y_raw":        round(float(row["policy_risk_score_weighted"]), 4),
            "baseline_avg": np.nan,   # not applicable
            "commodity":    commodity,
            "indicator":    "policy_risk_weighted",
            "data_period":  "baseline" if row["year_month"] <= BASELINE_END else "current"
        })

df_policy_ts = pd.DataFrame(policy_rows)
df_policy_ts = df_policy_ts.sort_values(["commodity","ds"]).reset_index(drop=True)

print(f"Policy time series: {df_policy_ts.shape}")
print(f"\nPolicy score stats:")
print(df_policy_ts.groupby("commodity")["y"].describe().round(2))

save_to_lakehouse(df_policy_ts, "prophet_ts_policy")

StatementMeta(, 9969517f-4145-4232-b5a7-9da711710490, 17, Finished, Available, Finished, False)


=== Step 7: Policy — rule-based (no 5Y avg) ===
Policy time series: (360, 7)

Policy score stats:
           count   mean    std  min  25%    50%    75%    max
commodity                                                    
Barley      72.0   0.00   0.00  0.0  0.0   0.00   0.00   0.00
Corn        72.0   2.69   2.35  0.0  0.0   3.38   4.92   6.42
Rice        72.0  21.70  26.57  0.0  0.0  14.34  32.51  92.80
Soybean     72.0   2.38   2.15  0.0  0.0   2.86   3.77   6.26
Wheat       72.0  13.99  12.58  0.0  0.0  14.57  22.48  37.92
✓ srm.prophet_ts_policy: 360 rows saved


In [16]:
# ══════════════════════════════════════════════════════════════════
# STEP 8: BASELINE SUMMARY TABLE
# Document all baseline values for reference
# Critical for stakeholder transparency
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 8: Baseline summary ===")

baseline_rows = []

for commodity in COMMODITIES:
    baseline_rows.append({
        "commodity":          commodity,
        "indicator":          "stu_ratio",
        "baseline_avg":       stu_baseline[commodity],
        "unit":               "% STU",
        "interpretation":     "5Y avg global stock-to-use ratio",
        "direction":          "deviation = current - baseline (negative = risk)"
    })
    baseline_rows.append({
        "commodity":          commodity,
        "indicator":          "bdi_ratio",
        "baseline_avg":       round(bdi_baseline_median, 2),
        "unit":               "BDI index",
        "interpretation":     "5Y avg BDI value (same for all commodities)",
        "direction":          "ratio = current / baseline (>1 = risk)"
    })
    baseline_rows.append({
        "commodity":          commodity,
        "indicator":          "demand_ratio",
        "baseline_avg":       demand_baseline[commodity],
        "unit":               "MT/month",
        "interpretation":     "5Y avg total tracked buyer import volume",
        "direction":          "ratio = current / baseline (>1 = risk)"
    })
    baseline_rows.append({
        "commodity":          commodity,
        "indicator":          "ppi_deviation",
        "baseline_avg":       ppi_baseline[commodity],
        "unit":               "PPI index",
        "interpretation":     "5Y avg production potential index",
        "direction":          "deviation = current - baseline (negative = risk)"
    })
    baseline_rows.append({
        "commodity":          commodity,
        "indicator":          "ksa_deviation",
        "baseline_avg":       ksa_baseline[commodity],
        "unit":               "% top3 share",
        "interpretation":     "5Y avg KSA top3 import concentration",
        "direction":          "deviation = current - baseline (positive = risk)"
    })
    baseline_rows.append({
        "commodity":          commodity,
        "indicator":          "fob_price_deviation",
        "baseline_avg":       price_baseline[commodity],
        "unit":               "USD/mt (or EUR/mt for Wheat if EUR series)",
        "interpretation":     "5Y avg FOB price, confirmed origin per commodity",
        "direction":          "deviation = current - baseline (positive = risk)"
    })

df_baseline = pd.DataFrame(baseline_rows)

print("\nBaseline values (Jan 2021 — Dec 2025):")
print(f"\n{'Commodity':<10} {'Indicator':<20} {'Baseline Avg':>14} {'Unit':<15}")
print("-" * 65)
for _, row in df_baseline.iterrows():
    print(f"{row['commodity']:<10} {row['indicator']:<20} {row['baseline_avg']:>14.3f} {row['unit']:<15}")

save_to_lakehouse(df_baseline, "prophet_baseline_summary")

StatementMeta(, 9969517f-4145-4232-b5a7-9da711710490, 18, Finished, Available, Finished, False)


=== Step 8: Baseline summary ===

Baseline values (Jan 2021 — Dec 2025):

Commodity  Indicator              Baseline Avg Unit           
-----------------------------------------------------------------
Wheat      stu_ratio                    26.862 % STU          
Wheat      bdi_ratio                  1483.740 BDI index      
Wheat      demand_ratio            5968140.430 MT/month       
Wheat      ppi_deviation               100.620 PPI index      
Wheat      ksa_deviation                81.856 % top3 share   
Wheat      fob_price_deviation         270.767 USD/mt (or EUR/mt for Wheat if EUR series)
Corn       stu_ratio                    21.425 % STU          
Corn       bdi_ratio                  1483.740 BDI index      
Corn       demand_ratio            6982974.530 MT/month       
Corn       ppi_deviation               101.010 PPI index      
Corn       ksa_deviation                96.762 % top3 share   
Corn       fob_price_deviation         233.825 USD/mt (or EUR/mt for Wheat i

In [17]:
# ══════════════════════════════════════════════════════════════════
# STEP 9: COMBINED TABLE
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 9: Building combined time series table ===")

df_all_ts = pd.concat([
    df_stu_ts[["ds","y","y_raw","baseline_avg","commodity","indicator","data_period"]],
    df_bdi_ts[["ds","y","y_raw","baseline_avg","commodity","indicator","data_period"]],
    df_demand_ts[["ds","y","y_raw","baseline_avg","commodity","indicator","data_period"]],
    df_ppi_ts[["ds","y","y_raw","baseline_avg","commodity","indicator","data_period"]],
    df_ksa_ts[["ds","y","y_raw","baseline_avg","commodity","indicator","data_period"]],
    df_policy_ts[["ds","y","y_raw","baseline_avg","commodity","indicator","data_period"]],
    df_price_ts[["ds","y","y_raw","baseline_avg","commodity","indicator","data_period"]],   # NEW
], ignore_index=True)

df_all_ts = df_all_ts.sort_values(
    ["commodity","indicator","ds"]
).reset_index(drop=True)

save_to_lakehouse(df_all_ts, "prophet_ts_all")

print(f"Combined table: {df_all_ts.shape}")

StatementMeta(, 9969517f-4145-4232-b5a7-9da711710490, 19, Finished, Available, Finished, False)


=== Step 9: Building combined time series table ===
✓ srm.prophet_ts_all: 2617 rows saved
Combined table: (2617, 7)


In [18]:
# ══════════════════════════════════════════════════════════════════
# STEP 10: VALIDATION SUMMARY
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 10: Validation summary ===")

print(f"\n{'Indicator':<22} {'Commodity':<10} {'Baseline':>10} {'Current':>10} {'Signal':>12} {'Risk Direction'}")
print("-" * 80)

current_month = SCORING_END

indicator_map = {
    "stu_deviation":      df_stu_ts,
    "bdi_ratio":          df_bdi_ts,
    "demand_ratio":       df_demand_ts,
    "ppi_deviation":      df_ppi_ts,
    "ksa_deviation":      df_ksa_ts,
    "policy_risk_weighted": df_policy_ts,
    "fob_price_deviation":  df_price_ts   # NEW
}

for indicator, df_ts in indicator_map.items():
    for commodity in COMMODITIES:
        dc = df_ts[df_ts["commodity"]==commodity]
        current_row = dc[dc["ds"] <= current_month].sort_values("ds").tail(1)
        if current_row.empty:
            continue

        y_current  = current_row["y"].values[0]
        y_raw      = current_row["y_raw"].values[0]
        baseline   = current_row["baseline_avg"].values[0] if "baseline_avg" in current_row.columns else np.nan

        if indicator in ["stu_deviation","ppi_deviation"]:
            risk_dir = "↓ Risk" if y_current < 0 else "✓ OK"
            signal   = f"{y_current:+.2f}"
        # elif indicator == "fob_price_deviation":                      # NEW
        #     risk_dir = "↑ Risk" if y_current > 0 else "✓ OK"          # NEW — opposite of STU/PPI
        #     signal   = f"{y_current:+.2f}"                            # NEW
        elif indicator == "fob_price_deviation":
            price_dev = y_raw - baseline    # CHANGED — use y_raw (actual price) and baseline, not y_current (now log-price)
            risk_dir = "↑ Risk" if price_dev > 0 else "✓ OK"
            signal   = f"{price_dev:+.2f}"
        elif indicator in ["bdi_ratio","demand_ratio"]:
            risk_dir = "↑ Risk" if y_current > 1.0 else "✓ OK"
            signal   = f"{y_current:.3f}x"
        elif indicator == "ksa_deviation":
            risk_dir = "↑ Risk" if y_current > 0 else "✓ OK"
            signal   = f"{y_current:+.2f}%"
        else:
            risk_dir = "↑ Risk" if y_current > 30 else "✓ OK"
            signal   = f"{y_current:.1f}"

        print(f"{indicator:<22} {commodity:<10} {baseline:>10.2f} {y_raw:>10.2f} {signal:>12} {risk_dir}")

print(f"""
Signal interpretation:
  STU deviation:  negative = stocks below 5Y avg = risk
  BDI ratio:      > 1.0    = freight above 5Y avg = risk
  Demand ratio:   > 1.0    = imports above 5Y avg = risk
  PPI deviation:  negative = production below 5Y avg = risk
  KSA deviation:  positive = more concentrated than 5Y avg = risk
  Policy:         > 30     = active restrictions = risk
  FOB Price:      positive = price above 5Y avg = risk        # NEW

Baseline period: Jan 2021 → Dec 2025 (fixed)
""")

print("=== NOTEBOOK 1 COMPLETE ===")
print("Next: Notebook 2 — Prophet Models (retrain with new signal definitions)")

StatementMeta(, 9969517f-4145-4232-b5a7-9da711710490, 20, Finished, Available, Finished, False)


=== Step 10: Validation summary ===

Indicator              Commodity    Baseline    Current       Signal Risk Direction
--------------------------------------------------------------------------------
stu_deviation          Wheat           26.86      26.25        -0.61 ↓ Risk
stu_deviation          Corn            21.42      17.99        -3.43 ↓ Risk
stu_deviation          Rice            30.84      31.80        +0.95 ✓ OK
stu_deviation          Soybean         19.63      19.64        +0.01 ✓ OK
stu_deviation          Barley          12.15      13.46        +1.31 ✓ OK
bdi_ratio              Wheat         1483.74    2769.91       1.527x ↑ Risk
bdi_ratio              Corn          1483.74    2769.91       1.527x ↑ Risk
bdi_ratio              Rice          1483.74    2769.91       1.527x ↑ Risk
bdi_ratio              Soybean       1483.74    2769.91       1.527x ↑ Risk
bdi_ratio              Barley        1483.74    2769.91       1.527x ↑ Risk
demand_ratio           Wheat      5968140.4